In [1]:
import numpy

In [2]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_chroma import Chroma  # Chroma = the vector database that stores and searches our document embeddings

In [ ]:
# These are just plain documents for now (text + metadata).
# They will later be converted into vectors and stored in Chroma.
from langchain_core.documents import Document

doc1 = Document(
    page_content = "Virat Kohli is renowned for his aggressive batting style and has captained the Royal Challengers Bangalore in the IPL",
    metadata = {
        "team": "RCB"
    }
)
doc2 = Document(
    page_content = "Rohit Sharma is famous for holding the record for the highest individual score in ODI cricket while playing for Mumbai Indians.",
    metadata = {
        "team": "MI"
    }
)

doc3 = Document(
    page_content = "Babar Azam is regarded as one of the most elegant batsmen of his generation and represents Pakistan internationally.",
    metadata = {
        "team": "SRH"
    }
)

doc4 = Document(
    page_content = "Ben Stokes is celebrated for his all-round match-winning performances and leads England's Test team.",
    metadata = {
        "team": "RCB"
    }
)

In [13]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Set up embeddings (local, free, no API key)
# This is the "embedding model" - it turns text into a list of numbers (a vector).
# Chroma uses these vectors to figure out which documents are similar to each other.
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "d:\langchain_models\venv\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [5]:
# Collect all documents into one list so we can add them to the vector store in one go
docs = [doc1, doc2, doc3, doc4]

In [9]:
# Create (or connect to) a Chroma vector store
# embedding_function -> the model used to turn text into vectors (see note below)
# persist_directory  -> the folder on disk where the vector data gets saved
# collection_name    -> name for this set of vectors, kind of like a table name
# NOTE: below it says `embedding_model`, but earlier the variable was named `embeddings`.
# Make sure this name matches what you defined above, or it will throw a NameError.
vector_store = Chroma(
    embedding_function= embedding_model,
    persist_directory= "chroma_db",
    collection_name= "sample"
)

In [ ]:
# Add our documents into the vector store.
# Chroma will use the embedding model to turn each document's text into a vector,
# then save that vector + its metadata so it can be searched later.
# vector_store is a dict
vector_store.add_documents(documents = docs)

GoogleGenerativeAIError: Error embedding content (UNAUTHENTICATED): 401 UNAUTHENTICATED. {'error': {'code': 401, 'message': 'Request had invalid authentication credentials. Expected OAuth 2 access token, login cookie or other valid authentication credential. See https://developers.google.com/identity/sign-in/web/devconsole-project.', 'status': 'UNAUTHENTICATED', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'ACCESS_TOKEN_TYPE_UNSUPPORTED', 'metadata': {'service': 'generativelanguage.googleapis.com', 'method': 'google.ai.generativelanguage.v1beta.GenerativeService.BatchEmbedContents'}}]}}

In [ ]:
# Fetch everything currently saved in the vector store.
# include=["metadatas", "documents", "embeddings"] means:
# also show the metadata, the original text, and the raw vectors (numbers) for each entry.
vector_store.get(include = ["metadatas", "documents", "embeddings"])

In [ ]:
# Search documnets
# Ask the vector store: "which stored documents are closest in meaning to this query?"
# k=2 means: give me the top 2 most similar results.
vector_store.similarity_search(query = "Who among this are bowlers?",
                                k = 2
                               )

In [ ]:
# Embedding of 1st doc
# Take just the first vector out of a list of embeddings
embed = embed[0]

In [ ]:
# Return docs most similar to embedding vector.
# Same idea as similarity_search, but this time we already have a vector (numbers)
# instead of raw text, and we ask Chroma to find the closest matching stored vectors.
vector_store.similarity_search_by_vector(embeddings = embed,
                                        k = 2)

In [ ]:
# Search the vector store and also get a similarity score back.
# filter={"team": "RCB"} means: only look inside documents whose metadata
# "team" field equals "RCB" before doing the similarity search.
vector_store.similarity_search_with_scores(
    query = "",
    k = 2,
    filter = {"team": "RCB"}
)

In [ ]:
update_doc1 = Document(
    page_content = "Virat Kohli is widely regarded as one of the greatest ODI batsmen of all time, holding the record for most centuries in the format. He has also played a key role in India's rise to the top of Test cricket rankings during his tenure as captain.",
    metadata = {
        "team": "RCB"
    }
)

# Replace an existing document already stored in the vector store with a new version.
# document_id -> the id of the document you want to overwrite
# document    -> the new content/metadata that should replace it
vector_store.update_document(document_id = document_id, document = update_doc1)

In [ ]:
# Remove document(s) from the vector store using their id(s)
vector_store.delete(ids = [ids])